In [1]:
#PART A

import pandas as pd
import numpy as np
import time

# 1. Load the dataset
df = pd.read_csv('garments_worker_productivity.csv')

# 2. Clean 'department' whitespace and typo ('sweing' -> 'sewing')
df['department'] = df['department'].str.strip().replace({'sweing': 'sewing'})

# 3. Handle missing values in 'wip' (Work In Progress)
# Missing WIP occurs mainly in finishing lines where WIP is not tracked -> fill with 0
df['wip'] = df['wip'].fillna(0)

# 4. Create Classification Target: MeetsTarget (1 if actual >= targeted, else 0)
df['MeetsTarget'] = (df['actual_productivity'] >= df['targeted_productivity']).astype(int)

# 5. Drop 'date' (we already have 'day' and 'quarter')
if 'date' in df.columns:
    df = df.drop(columns=['date'])

print("Cleaned Data Shape:", df.shape)
print("\nTarget Class Distribution (MeetsTarget):")
print(df['MeetsTarget'].value_counts())
df.head(3)

Cleaned Data Shape: (1197, 15)

Target Class Distribution (MeetsTarget):
MeetsTarget
1    875
0    322
Name: count, dtype: int64


,quarter,department,day,team,targeted_productivity,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,no_of_workers,actual_productivity,MeetsTarget
0,Quarter1,sewing,Thursday,8,0.80,26.16,1108.0,7080,98,0.0,0,0,59.0,0.940725,1
1,Quarter1,finishing,Thursday,1,0.75,3.94,0.0,960,0,0.0,0,0,8.0,0.886500,1
2,Quarter1,sewing,Thursday,11,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570,1


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Separate Features (X) and Targets (y_reg for regression, y_clf for classification)
# Notice: 'actual_productivity' and 'MeetsTarget' are both dropped from X!
X = df.drop(columns=['actual_productivity', 'MeetsTarget'])
y_reg = df['actual_productivity']
y_clf = df['MeetsTarget']

# Identify categorical and numerical columns
categorical_cols = ['quarter', 'department', 'day', 'team']
numerical_cols = [col for col in X.columns if col not in categorical_cols]

# Create preprocessor: OneHot for categories, StandardScaler for numbers
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_cols)
    ]
)

# Fixed Train-Test Split (80% Train, 20% Test)
# We split X, y_reg, and y_clf simultaneously using the same random_state
X_train_raw, X_test_raw, y_train_reg, y_test_reg, y_train_clf, y_test_clf = train_test_split(
    X, y_reg, y_clf, test_size=0.2, random_state=42
)

# Fit preprocessor ONLY on training data, then transform both train and test (prevents data snooping!)
X_train = preprocessor.fit_transform(X_train_raw)
X_test = preprocessor.transform(X_test_raw)

print(f"Training feature shape: {X_train.shape}")
print(f"Testing feature shape: {X_test.shape}")

Training feature shape: (957, 30)
Testing feature shape: (240, 30)


In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Initialize Model
sklearn_lr = LinearRegression()

# 2. Train and measure training time
start_train = time.perf_counter()
sklearn_lr.fit(X_train, y_train_reg)
sklearn_lr_train_time = time.perf_counter() - start_train

# 3. Predict and measure prediction time
start_pred = time.perf_counter()
y_pred_reg_sklearn = sklearn_lr.predict(X_test)
sklearn_lr_pred_time = time.perf_counter() - start_pred

# 4. Evaluate Metrics
mae_sklearn = mean_absolute_error(y_test_reg, y_pred_reg_sklearn)
rmse_sklearn = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg_sklearn))
r2_sklearn = r2_score(y_test_reg, y_pred_reg_sklearn)

print("--- Scikit-Learn Linear Regression Results ---")
print(f"Training Time   : {sklearn_lr_train_time:.6f} seconds")
print(f"Prediction Time : {sklearn_lr_pred_time:.6f} seconds")
print(f"MAE             : {mae_sklearn:.4f}")
print(f"RMSE            : {rmse_sklearn:.4f}")
print(f"R2 Score        : {r2_sklearn:.4f}")

--- Scikit-Learn Linear Regression Results ---
Training Time   : 0.026809 seconds
Prediction Time : 0.001491 seconds
MAE             : 0.1106
RMSE            : 0.1495
R2 Score        : 0.1584


In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. Initialize Model
sklearn_logreg = LogisticRegression(max_iter=1000, random_state=42)

# 2. Train and measure training time
start_train = time.perf_counter()
sklearn_logreg.fit(X_train, y_train_clf)
sklearn_logreg_train_time = time.perf_counter() - start_train

# 3. Predict and measure prediction time
start_pred = time.perf_counter()
y_pred_clf_sklearn = sklearn_logreg.predict(X_test)
sklearn_logreg_pred_time = time.perf_counter() - start_pred

# 4. Evaluate Metrics
acc_sklearn = accuracy_score(y_test_clf, y_pred_clf_sklearn)
prec_sklearn = precision_score(y_test_clf, y_pred_clf_sklearn)
rec_sklearn = recall_score(y_test_clf, y_pred_clf_sklearn)
f1_sklearn = f1_score(y_test_clf, y_pred_clf_sklearn)

print("--- Scikit-Learn Logistic Regression Results ---")
print(f"Training Time   : {sklearn_logreg_train_time:.6f} seconds")
print(f"Prediction Time : {sklearn_logreg_pred_time:.6f} seconds")
print(f"Accuracy        : {acc_sklearn:.4f}")
print(f"Precision       : {prec_sklearn:.4f}")
print(f"Recall          : {rec_sklearn:.4f}")
print(f"F1 Score        : {f1_sklearn:.4f}")

--- Scikit-Learn Logistic Regression Results ---
Training Time   : 0.042235 seconds
Prediction Time : 0.000643 seconds
Accuracy        : 0.7583
Precision       : 0.7874
Recall          : 0.9209
F1 Score        : 0.8490


In [5]:
#PART B

# ==========================================
# PART B - STEP 1: MANUAL PREPROCESSING
# ==========================================

# 1. One-Hot Encoding using pd.get_dummies (drop_first=True)
# We concatenate train and test temporarily to guarantee matching dummy columns,
# but we do NOT leak any values or statistics!
n_train = len(X_train_raw)
combined_raw = pd.concat([X_train_raw, X_test_raw], axis=0)

# Convert categorical columns to dummies
combined_encoded = pd.get_dummies(combined_raw, columns=categorical_cols, drop_first=True, dtype=float)

# Split back to train and test
X_train_manual = combined_encoded.iloc[:n_train].copy()
X_test_manual = combined_encoded.iloc[n_train:].copy()

# 2. Manual StandardScaler on Numerical Columns
# Compute mean and std on TRAINING SET ONLY
train_means = X_train_manual[numerical_cols].mean()
train_stds = X_train_manual[numerical_cols].std()

# Handle any constant column with std=0 to prevent division by zero
train_stds = train_stds.replace(0, 1.0)

# Apply Z-score standardization: (x - mu) / sigma
X_train_manual[numerical_cols] = (X_train_manual[numerical_cols] - train_means) / train_stds
X_test_manual[numerical_cols] = (X_test_manual[numerical_cols] - train_means) / train_stds

# Convert to pure NumPy arrays
X_tr_np = X_train_manual.values
X_te_np = X_test_manual.values
y_tr_reg_np = y_train_reg.values
y_te_reg_np = y_test_reg.values
y_tr_clf_np = y_train_clf.values
y_te_clf_np = y_test_clf.values

print(f"Manual preprocessed train shape: {X_tr_np.shape}")
print(f"Manual preprocessed test shape : {X_te_np.shape}")

Manual preprocessed train shape: (957, 30)
Manual preprocessed test shape : (240, 30)


In [6]:
# ==========================================
# PART B - STEP 2: MANUAL EVALUATION METRICS
# ==========================================

# Regression Metrics
def manual_mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def manual_rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def manual_r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ss_res / ss_tot)

# Classification Metrics
def manual_confusion_matrix(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    return tp, tn, fp, fn

def manual_accuracy(y_true, y_pred):
    tp, tn, fp, fn = manual_confusion_matrix(y_true, y_pred)
    return (tp + tn) / (tp + tn + fp + fn)

def manual_precision(y_true, y_pred):
    tp, tn, fp, fn = manual_confusion_matrix(y_true, y_pred)
    return tp / (tp + fp) if (tp + fp) > 0 else 0.0

def manual_recall(y_true, y_pred):
    tp, tn, fp, fn = manual_confusion_matrix(y_true, y_pred)
    return tp / (tp + fn) if (tp + fn) > 0 else 0.0

def manual_f1(y_true, y_pred):
    p = manual_precision(y_true, y_pred)
    r = manual_recall(y_true, y_pred)
    return 2 * (p * r) / (p + r) if (p + r) > 0 else 0.0

print("Manual metrics defined successfully.")

Manual metrics defined successfully.


In [7]:
# ==========================================
# PART B - STEP 3: MANUAL LINEAR REGRESSION
# ==========================================

class ManualLinearRegression:
    def __init__(self):
        self.weights = None
        
    def fit(self, X, y):
        # Add bias column (vector of ones)
        ones = np.ones((X.shape[0], 1))
        X_b = np.hstack([ones, X])
        
        # Closed-form solution: w = (X^T * X)^(-1) * X^T * y
        # We use np.linalg.pinv for numerical stability against collinearity
        self.weights = np.linalg.pinv(X_b.T @ X_b) @ X_b.T @ y
        
    def predict(self, X):
        ones = np.ones((X.shape[0], 1))
        X_b = np.hstack([ones, X])
        return X_b @ self.weights

# 1. Initialize and measure training time
manual_lr = ManualLinearRegression()
start_train = time.perf_counter()
manual_lr.fit(X_tr_np, y_tr_reg_np)
manual_lr_train_time = time.perf_counter() - start_train

# 2. Measure prediction time
start_pred = time.perf_counter()
y_pred_reg_manual = manual_lr.predict(X_te_np)
manual_lr_pred_time = time.perf_counter() - start_pred

# 3. Calculate manual evaluation metrics
mae_manual = manual_mae(y_te_reg_np, y_pred_reg_manual)
rmse_manual = manual_rmse(y_te_reg_np, y_pred_reg_manual)
r2_manual = manual_r2(y_te_reg_np, y_pred_reg_manual)

print("--- Manual Linear Regression Results ---")
print(f"Training Time   : {manual_lr_train_time:.6f} seconds")
print(f"Prediction Time : {manual_lr_pred_time:.6f} seconds")
print(f"MAE             : {mae_manual:.4f}")
print(f"RMSE            : {rmse_manual:.4f}")
print(f"R2 Score        : {r2_manual:.4f}")

--- Manual Linear Regression Results ---
Training Time   : 0.017702 seconds
Prediction Time : 0.000144 seconds
MAE             : 0.1106
RMSE            : 0.1495
R2 Score        : 0.1584


In [8]:
# ==========================================
# PART B - STEP 4: MANUAL LOGISTIC REGRESSION
# ==========================================

class ManualLogisticRegression:
    def __init__(self, learning_rate=0.1, n_iterations=1000):
        self.lr = learning_rate
        self.n_iterations = n_iterations
        self.weights = None
        
    def _sigmoid(self, z):
        # Clip z to prevent numerical overflow in exp(-z)
        z = np.clip(z, -500, 500)
        return 1.0 / (1.0 + np.exp(-z))
    
    def fit(self, X, y):
        m, n = X.shape
        # Add bias column
        ones = np.ones((m, 1))
        X_b = np.hstack([ones, X])
        
        # Initialize weights with zeros
        self.weights = np.zeros(n + 1)
        
        # Vectorized Gradient Descent
        for _ in range(self.n_iterations):
            linear_output = X_b @ self.weights
            probabilities = self._sigmoid(linear_output)
            
            # Gradient: (1/m) * X^T * (p - y)
            gradients = (1 / m) * (X_b.T @ (probabilities - y))
            
            # Update weights
            self.weights -= self.lr * gradients
            
    def predict_proba(self, X):
        ones = np.ones((X.shape[0], 1))
        X_b = np.hstack([ones, X])
        return self._sigmoid(X_b @ self.weights)
    
    def predict(self, X, threshold=0.5):
        probabilities = self.predict_proba(X)
        return (probabilities >= threshold).astype(int)

# 1. Initialize and measure training time
manual_logreg = ManualLogisticRegression(learning_rate=0.1, n_iterations=1500)
start_train = time.perf_counter()
manual_logreg.fit(X_tr_np, y_tr_clf_np)
manual_logreg_train_time = time.perf_counter() - start_train

# 2. Measure prediction time
start_pred = time.perf_counter()
y_pred_clf_manual = manual_logreg.predict(X_te_np)
manual_logreg_pred_time = time.perf_counter() - start_pred

# 3. Calculate manual evaluation metrics
acc_manual = manual_accuracy(y_te_clf_np, y_pred_clf_manual)
prec_manual = manual_precision(y_te_clf_np, y_pred_clf_manual)
rec_manual = manual_recall(y_te_clf_np, y_pred_clf_manual)
f1_manual = manual_f1(y_te_clf_np, y_pred_clf_manual)

print("--- Manual Logistic Regression Results ---")
print(f"Training Time   : {manual_logreg_train_time:.6f} seconds")
print(f"Prediction Time : {manual_logreg_pred_time:.6f} seconds")
print(f"Accuracy        : {acc_manual:.4f}")
print(f"Precision       : {prec_manual:.4f}")
print(f"Recall          : {rec_manual:.4f}")
print(f"F1 Score        : {f1_manual:.4f}")

--- Manual Logistic Regression Results ---
Training Time   : 0.111230 seconds
Prediction Time : 0.000272 seconds
Accuracy        : 0.7667
Precision       : 0.7788
Recall          : 0.9548
F1 Score        : 0.8579


In [9]:
# ==========================================
# PART C - STEP 1: COMPARISON TABLES
# ==========================================

# 1. Regression Comparison Table
reg_comparison = pd.DataFrame({
    'Implementation': ['Scikit-learn', 'Manual (From Scratch)'],
    'MAE': [mae_sklearn, mae_manual],
    'RMSE': [rmse_sklearn, rmse_manual],
    'R2 Score': [r2_sklearn, r2_manual],
    'Train Time (s)': [sklearn_lr_train_time, manual_lr_train_time],
    'Pred Time (s)': [sklearn_lr_pred_time, manual_lr_pred_time]
})

print("=================== REGRESSION COMPARISON ===================")
display(reg_comparison)

# 2. Classification Comparison Table
clf_comparison = pd.DataFrame({
    'Implementation': ['Scikit-learn', 'Manual (From Scratch)'],
    'Accuracy': [acc_sklearn, acc_manual],
    'Precision': [prec_sklearn, prec_manual],
    'Recall': [rec_sklearn, rec_manual],
    'F1 Score': [f1_sklearn, f1_manual],
    'Train Time (s)': [sklearn_logreg_train_time, manual_logreg_train_time],
    'Pred Time (s)': [sklearn_logreg_pred_time, manual_logreg_pred_time]
})

print("\n================= CLASSIFICATION COMPARISON =================")
display(clf_comparison)

=================== REGRESSION COMPARISON ===================


,Implementation,MAE,RMSE,R2 Score,Train Time (s),Pred Time (s)
0,Scikit-learn,0.110626,0.149485,0.158426,0.026809,0.001491
1,Manual (From Scratch),0.110626,0.149485,0.158426,0.017702,0.000144



================= CLASSIFICATION COMPARISON =================


,Implementation,Accuracy,Precision,Recall,F1 Score,Train Time (s),Pred Time (s)
0,Scikit-learn,0.758333,0.787440,0.920904,0.848958,0.042235,0.000643
1,Manual (From Scratch),0.766667,0.778802,0.954802,0.857868,0.111230,0.000272


In [10]:
# ==========================================
# PART C - STEP 2: OPTIMIZED MANUAL MODELS
# ==========================================

# 1. Optimized Linear Regression (Ridge / L2 Regularization)
class OptimizedLinearRegression:
    def __init__(self, l2_reg=0.1):
        self.l2_reg = l2_reg
        self.weights = None
        
    def fit(self, X, y):
        m, n = X.shape
        ones = np.ones((m, 1))
        X_b = np.hstack([ones, X])
        
        # Regularization matrix: don't regularize bias (index 0)
        I = np.eye(n + 1)
        I[0, 0] = 0.0
        
        # w = (X^T * X + lambda * I)^(-1) * X^T * y
        self.weights = np.linalg.pinv(X_b.T @ X_b + self.l2_reg * I) @ X_b.T @ y
        
    def predict(self, X):
        ones = np.ones((X.shape[0], 1))
        X_b = np.hstack([ones, X])
        return X_b @ self.weights

# 2. Optimized Logistic Regression (Vectorized + L2 Regularization + Convergence Stopping)
class OptimizedLogisticRegression:
    def __init__(self, learning_rate=0.2, max_iter=2000, tol=1e-5, l2_reg=0.01):
        self.lr = learning_rate
        self.max_iter = max_iter
        self.tol = tol
        self.l2_reg = l2_reg
        self.weights = None
        self.iterations_run = 0
        
    def _sigmoid(self, z):
        z = np.clip(z, -500, 500)
        return 1.0 / (1.0 + np.exp(-z))
    
    def fit(self, X, y):
        m, n = X.shape
        ones = np.ones((m, 1))
        X_b = np.hstack([ones, X])
        self.weights = np.zeros(n + 1)
        
        for i in range(self.max_iter):
            probs = self._sigmoid(X_b @ self.weights)
            
            # L2 penalty on weights (excluding bias at index 0)
            reg_term = (self.l2_reg / m) * self.weights
            reg_term[0] = 0.0
            
            gradient = (1 / m) * (X_b.T @ (probs - y)) + reg_term
            step = self.lr * gradient
            self.weights -= step
            
            # Check convergence (early stopping when change is smaller than tol)
            if np.linalg.norm(step) < self.tol:
                self.iterations_run = i + 1
                break
        else:
            self.iterations_run = self.max_iter
            
    def predict_proba(self, X):
        ones = np.ones((X.shape[0], 1))
        X_b = np.hstack([ones, X])
        return self._sigmoid(X_b @ self.weights)
    
    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

# Train and measure Optimized Linear Regression
opt_lr = OptimizedLinearRegression(l2_reg=0.05)
start_train = time.perf_counter()
opt_lr.fit(X_tr_np, y_tr_reg_np)
opt_lr_train_time = time.perf_counter() - start_train

start_pred = time.perf_counter()
y_pred_reg_opt = opt_lr.predict(X_te_np)
opt_lr_pred_time = time.perf_counter() - start_pred

# Train and measure Optimized Logistic Regression
opt_logreg = OptimizedLogisticRegression(learning_rate=0.2, max_iter=2000, tol=1e-5, l2_reg=0.01)
start_train = time.perf_counter()
opt_logreg.fit(X_tr_np, y_tr_clf_np)
opt_logreg_train_time = time.perf_counter() - start_train

start_pred = time.perf_counter()
y_pred_clf_opt = opt_logreg.predict(X_te_np)
opt_logreg_pred_time = time.perf_counter() - start_pred

print("Optimized models trained successfully!")
print(f"Optimized Logistic Regression stopped early at iteration: {opt_logreg.iterations_run}")

Optimized models trained successfully!
Optimized Logistic Regression stopped early at iteration: 2000


In [11]:
# ==========================================
# PART C - STEP 3: FINAL THREE-WAY COMPARISON
# ==========================================

final_reg_df = pd.DataFrame({
    'Model': ['Sklearn LinearRegression', 'Manual Scratch', 'Manual Optimized (Ridge)'],
    'MAE': [mae_sklearn, mae_manual, manual_mae(y_te_reg_np, y_pred_reg_opt)],
    'RMSE': [rmse_sklearn, rmse_manual, manual_rmse(y_te_reg_np, y_pred_reg_opt)],
    'R2 Score': [r2_sklearn, r2_manual, manual_r2(y_te_reg_np, y_pred_reg_opt)],
    'Train Time (s)': [sklearn_lr_train_time, manual_lr_train_time, opt_lr_train_time],
    'Pred Time (s)': [sklearn_lr_pred_time, manual_lr_pred_time, opt_lr_pred_time]
})

final_clf_df = pd.DataFrame({
    'Model': ['Sklearn LogisticRegression', 'Manual Scratch', 'Manual Optimized (L2+EarlyStop)'],
    'Accuracy': [acc_sklearn, acc_manual, manual_accuracy(y_te_clf_np, y_pred_clf_opt)],
    'Precision': [prec_sklearn, prec_manual, manual_precision(y_te_clf_np, y_pred_clf_opt)],
    'Recall': [rec_sklearn, rec_manual, manual_recall(y_te_clf_np, y_pred_clf_opt)],
    'F1 Score': [f1_sklearn, f1_manual, manual_f1(y_te_clf_np, y_pred_clf_opt)],
    'Train Time (s)': [sklearn_logreg_train_time, manual_logreg_train_time, opt_logreg_train_time],
    'Pred Time (s)': [sklearn_logreg_pred_time, manual_logreg_pred_time, opt_logreg_pred_time]
})

print("--- FINAL REGRESSION BENCHMARK ---")
display(final_reg_df)

print("\n--- FINAL CLASSIFICATION BENCHMARK ---")
display(final_clf_df)

--- FINAL REGRESSION BENCHMARK ---


,Model,MAE,RMSE,R2 Score,Train Time (s),Pred Time (s)
0,Sklearn LinearRegression,0.110626,0.149485,0.158426,0.026809,0.001491
1,Manual Scratch,0.110626,0.149485,0.158426,0.017702,0.000144
2,Manual Optimized (Ridge),0.110588,0.149459,0.158722,0.004676,0.000344



--- FINAL CLASSIFICATION BENCHMARK ---


,Model,Accuracy,Precision,Recall,F1 Score,Train Time (s),Pred Time (s)
0,Sklearn LogisticRegression,0.758333,0.787440,0.920904,0.848958,0.042235,0.000643
1,Manual Scratch,0.766667,0.778802,0.954802,0.857868,0.111230,0.000272
2,Manual Optimized (L2+EarlyStop),0.758333,0.790244,0.915254,0.848168,0.142917,0.000201
